In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/Early-Sepsis-Detection"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Early-Sepsis-Detection


In [ ]:
!pip install xgboost lightgbm catboost -q
!pip install optuna -q

# 13 — Hyperparameter Tuning
### Early Sepsis Detection — Phase 15

Tunes **LightGBM** (best PR-AUC in Phase 14) and **CatBoost** (best ROC-AUC
and recall in Phase 14) using Optuna, optimizing **validation PR-AUC**
(not accuracy — see Phase 12's justification).

**The test set remains completely untouched in this notebook** — all
tuning decisions are made using only the training set (for fitting) and
validation set (for scoring each trial).


In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import optuna
import lightgbm as lgb
import catboost as cb
from sklearn.metrics import average_precision_score

from src import config
from src import evaluate as ev

optuna.logging.set_verbosity(optuna.logging.WARNING)

npz = np.load(config.PROCESSED_DIR / "model_ready_arrays.npz")
X_train, y_train = npz["X_train"], npz["y_train"]
X_val, y_val = npz["X_val"], npz["y_val"]

n_pos = int(y_train.sum()); n_neg = len(y_train) - n_pos
scale_pos_weight = n_neg / n_pos
print(f"Tuning against validation PR-AUC. scale_pos_weight={scale_pos_weight:.4f}")


Tuning against validation PR-AUC. scale_pos_weight=14.9861


## 1. Tune LightGBM

Search space matches the project's specified LightGBM parameters:
`n_estimators, learning_rate, num_leaves, max_depth, min_child_samples,
subsample, colsample_bytree`.


In [ ]:
N_TRIALS = 30  # kept modest for CPU-only Colab runtime; increase if time allows

def lgb_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 127),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "scale_pos_weight": scale_pos_weight,
        "random_state": config.RANDOM_SEED,
        "n_jobs": -1,
        "verbose": -1,
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)
    val_prob = model.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, val_prob)

lgb_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=config.RANDOM_SEED))
lgb_study.optimize(lgb_objective, n_trials=N_TRIALS, show_progress_bar=True)

print("Best LightGBM validation PR-AUC:", lgb_study.best_value)
print("Best params:", lgb_study.best_params)


  0%|          | 0/30 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

Best LightGBM validation PR-AUC: 0.23968247068129184
Best params: {'n_estimators': 424, 'learning_rate': 0.0249064396938244, 'num_leaves': 26, 'max_depth': 8, 'min_child_samples': 47, 'subsample': 0.6488152939379115, 'colsample_bytree': 0.798070764044508}


## 2. Retrain final tuned LightGBM on full training set and evaluate

In [ ]:
best_lgb_params = {**lgb_study.best_params, "scale_pos_weight": scale_pos_weight,
                    "random_state": config.RANDOM_SEED, "n_jobs": -1, "verbose": -1}
lgb_tuned = lgb.LGBMClassifier(**best_lgb_params)
lgb_tuned.fit(X_train, y_train)

lgb_tuned_val_prob = lgb_tuned.predict_proba(X_val)[:, 1]
lgb_tuned_metrics = ev.compute_metrics(y_val, lgb_tuned_val_prob)
ev.print_metrics(lgb_tuned_metrics, "LightGBM (tuned, val)")
ev.save_model_results("LightGBM_tuned_val", lgb_tuned_metrics)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


--- LightGBM (tuned, val) (threshold=0.5) ---
  ROC-AUC:            0.7610
  PR-AUC:             0.2397
  Precision:          0.1797
  Recall (Sensitivity): 0.4905
  Specificity:        0.8508
  F1:                 0.2631
  Brier score:        0.1269
  Confusion matrix:   TN=4710 FP=826 FN=188 TP=181


2026-09-14 22:06:16,226 | INFO | src.evaluate | Saved/updated results for 'LightGBM_tuned_val' in /content/drive/MyDrive/Early-Sepsis-Detection/reports/tables/model_comparison.csv
INFO:src.evaluate:Saved/updated results for 'LightGBM_tuned_val' in /content/drive/MyDrive/Early-Sepsis-Detection/reports/tables/model_comparison.csv


## 3. Tune CatBoost

Search space uses CatBoost's native parameter names analogous to the
LightGBM set: `iterations, learning_rate, depth, l2_leaf_reg, subsample`.


In [ ]:
def cb_objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "depth": trial.suggest_int("depth", 3, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "auto_class_weights": "Balanced",
        "random_state": config.RANDOM_SEED,
        "verbose": False,
        "bootstrap_type": "Bernoulli",  # required for subsample to be used
    }
    model = cb.CatBoostClassifier(**params)
    model.fit(X_train, y_train)
    val_prob = model.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, val_prob)

cb_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=config.RANDOM_SEED))
cb_study.optimize(cb_objective, n_trials=N_TRIALS, show_progress_bar=True)

print("Best CatBoost validation PR-AUC:", cb_study.best_value)
print("Best params:", cb_study.best_params)


  0%|          | 0/30 [00:00<?, ?it/s]

Best CatBoost validation PR-AUC: 0.21889086632383054
Best params: {'iterations': 467, 'learning_rate': 0.04858160414190959, 'depth': 8, 'l2_leaf_reg': 7.076141620665087, 'subsample': 0.9476678356987474}


## 4. Retrain final tuned CatBoost and evaluate

In [ ]:
best_cb_params = {**cb_study.best_params, "auto_class_weights": "Balanced",
                   "random_state": config.RANDOM_SEED, "verbose": False,
                   "bootstrap_type": "Bernoulli"}
cb_tuned = cb.CatBoostClassifier(**best_cb_params)
cb_tuned.fit(X_train, y_train)

cb_tuned_val_prob = cb_tuned.predict_proba(X_val)[:, 1]
cb_tuned_metrics = ev.compute_metrics(y_val, cb_tuned_val_prob)
ev.print_metrics(cb_tuned_metrics, "CatBoost (tuned, val)")
ev.save_model_results("CatBoost_tuned_val", cb_tuned_metrics)


2026-09-15 00:10:34,654 | INFO | src.evaluate | Saved/updated results for 'CatBoost_tuned_val' in /content/drive/MyDrive/Early-Sepsis-Detection/reports/tables/model_comparison.csv
INFO:src.evaluate:Saved/updated results for 'CatBoost_tuned_val' in /content/drive/MyDrive/Early-Sepsis-Detection/reports/tables/model_comparison.csv


--- CatBoost (tuned, val) (threshold=0.5) ---
  ROC-AUC:            0.7394
  PR-AUC:             0.2189
  Precision:          0.2178
  Recall (Sensitivity): 0.3523
  Specificity:        0.9156
  F1:                 0.2692
  Brier score:        0.1034
  Confusion matrix:   TN=5069 FP=467 FN=239 TP=130


## 5. Before vs. after tuning comparison

**Did tuning actually help?** This is checked explicitly rather than
assumed — hyperparameter tuning does not always beat a reasonable default,
especially with a modest `N_TRIALS` budget.


In [ ]:
comparison = pd.DataFrame([
    {"model": "LightGBM (default, Phase 14)", **{k: v for k, v in
        pd.read_csv(config.TABLES_DIR / "model_comparison.csv")
        .set_index("model").loc["LightGBM_val"].to_dict().items() if k != "threshold"}},
    {"model": "LightGBM (tuned)", **{k: v for k, v in lgb_tuned_metrics.items() if k != "threshold"}},
    {"model": "CatBoost (default, Phase 14)", **{k: v for k, v in
        pd.read_csv(config.TABLES_DIR / "model_comparison.csv")
        .set_index("model").loc["CatBoost_val"].to_dict().items() if k != "threshold"}},
    {"model": "CatBoost (tuned)", **{k: v for k, v in cb_tuned_metrics.items() if k != "threshold"}},
])
display(comparison[["model", "roc_auc", "pr_auc", "recall_sensitivity", "specificity", "f1"]])


,model,roc_auc,pr_auc,recall_sensitivity,specificity,f1
0,"LightGBM (default, Phase 14)",0.710811,0.207377,0.298103,0.929010,0.252294
1,LightGBM (tuned),0.760980,0.239682,0.490515,0.850795,0.263081
2,"CatBoost (default, Phase 14)",0.722775,0.190043,0.365854,0.879335,0.230375
3,CatBoost (tuned),0.739415,0.218891,0.352304,0.915643,0.269151


## 6. Save tuned models and Optuna studies

In [ ]:
import joblib

joblib.dump(lgb_tuned, config.MODELS_DIR / "lightgbm_tuned.pkl")
cb_tuned.save_model(str(config.MODELS_DIR / "catboost_tuned.cbm"))

joblib.dump(lgb_study, config.MODELS_DIR / "lgb_optuna_study.pkl")
joblib.dump(cb_study, config.MODELS_DIR / "cb_optuna_study.pkl")

print("Saved tuned models and Optuna studies.")


Saved tuned models and Optuna studies.


---
### What to send back to Claude after running this notebook

- Section 1 & 3's best validation PR-AUC and best params for each model
- Section 5's before/after comparison table (did tuning help, and by how much?)

With that, **Phase 15 is complete**, and we move to **Phase 16 (Deep
Learning: LSTM/GRU)** — building a sequence model on the raw hourly time
series (not the patient-level aggregated features), to see whether a
model that sees the full temporal sequence can outperform these
feature-engineered tree models.
